# 1. Read the Dataset

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

#Include all the packages need..
import tensorflow as tf
import numpy as np
tf.random.set_seed(42)
import pandas as pd
from sklearn.model_selection import train_test_split
import os

from sklearn import preprocessing

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

In [2]:
#change working directory
#load the dataset
churn_df = pd.read_csv("bank.csv", na_values= '?')

#view top 10 rows
churn_df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Check if there any null values in the dataset
churn_df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [4]:
# Check the datatype of each attributes
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
RowNumber          10000 non-null int64
CustomerId         10000 non-null int64
Surname            10000 non-null object
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


# 2. Drop the columns which are unique for all users like IDs (5 points)

In [5]:
# Attributes like 'RowNumber', 'Surname' and 'CustomerId' can be removed because the unique values for each customers does not make any sence
churn_df = churn_df.drop(columns=['RowNumber', 'Surname', 'CustomerId'])

In [6]:
print("Values of the attribute 'Geography'is:", churn_df["Geography"].unique())
print("Values of the attribute 'Gender' is:", churn_df["Gender"].unique())

Values of the attribute 'Geography'is: ['France' 'Spain' 'Germany']
Values of the attribute 'Gender' is: ['Female' 'Male']


In [7]:
# Apply lable encoding for the attributes 'Geography' and 'Gender'
le = preprocessing.LabelEncoder()
churn_df["Geography"] = le.fit_transform(churn_df["Geography"])
churn_df["Gender"] = le.fit_transform(churn_df["Gender"])

In [8]:
churn_df.dtypes

CreditScore          int64
Geography            int64
Gender               int64
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [9]:
churn_df.groupby('Exited').count()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
Exited,,,,,,,,,,
0,7963,7963,7963,7963,7963,7963,7963,7963,7963,7963
1,2037,2037,2037,2037,2037,2037,2037,2037,2037,2037


In [10]:
churn_df.shape

(10000, 11)

# 3. Distinguish the feature and target set (5 points)

In [11]:
X = churn_df.drop('Exited', axis=1)
y = churn_df['Exited']

In [12]:
print("Feature set dimension:", X.shape)
print("Target data dimension:", y.shape)

Feature set dimension: (10000, 10)
Target data dimension: (10000,)


# 4. Divide the data set into Train and test sets

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, 
                                                    stratify=y, 
                                                    random_state=42)

# 5. Normalize the train and test data (5 points)

In [14]:
# Normalize the train the test data seperately because in the real world production data will the normalized seperately
X_train_scale = StandardScaler().fit_transform(X_train)
X_test_scale = StandardScaler().fit_transform(X_test)

/home/siju/.local/lib/python3.6/site-packages/sklearn/preprocessing/data.py:625: DataConversionWarning: Data with input dtype int64, float64 were all converted to float64 by StandardScaler.
  return self.partial_fit(X, y)
/home/siju/.local/lib/python3.6/site-packages/sklearn/base.py:462: DataConversionWarning: Data with input dtype int64, float64 were all converted to float64 by StandardScaler.
  return self.fit(X, **fit_params).transform(X)
/home/siju/.local/lib/python3.6/site-packages/sklearn/preprocessing/data.py:625: DataConversionWarning: Data with input dtype int64, float64 were all converted to float64 by StandardScaler.
  return self.partial_fit(X, y)
/home/siju/.local/lib/python3.6/site-packages/sklearn/base.py:462: DataConversionWarning: Data with input dtype int64, float64 were all converted to float64 by StandardScaler.
  return self.fit(X, **fit_params).transform(X)


In [15]:
X_train_scale.shape

(7000, 10)

In [16]:
y_train.shape

(7000,)

In [17]:
org_y_test = y_test

In [18]:
y_test = tf.keras.utils.to_categorical(y_test, num_classes=len(np.unique(y_test)))
y_train = tf.keras.utils.to_categorical(y_train, num_classes=len(np.unique(y_train)))

# 6. Initialize & build the model (10 points)

In [19]:
def create_model():
    #Initialize Sequential model
    model = tf.keras.models.Sequential()

    #Normalize the data 
    model.add(tf.keras.layers.BatchNormalization())

    #Add Dense Layer to create a hidden layer of 128 neurons with Relu activation
    model.add(tf.keras.layers.Dense(128, activation='relu'))

    #Add Dense Layer which provides 2 Outputs after applying softmax
    model.add(tf.keras.layers.Dense(2, activation='softmax'))

    #Create SDG optimizer
    sgd_optimizer = tf.keras.optimizers.SGD(lr=0.001)

    #Comile the model
    model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [20]:
model = create_model()
model.fit(X_train_scale, y_train,
          epochs=10,
          batch_size=32)

W0824 20:47:28.455522 140663360042816 deprecation.py:323] From /home/siju/.local/lib/python3.6/site-packages/tensorflow/python/ops/math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 0s 38us/sample - loss: 0.7706 - accuracy: 0.4636
Epoch 2/10
7000/7000 [==============================] - 0s 21us/sample - loss: 0.6278 - accuracy: 0.6693
Epoch 3/10
7000/7000 [==============================] - 0s 21us/sample - loss: 0.5644 - accuracy: 0.7530
Epoch 4/10
7000/7000 [==============================] - 0s 20us/sample - loss: 0.5324 - accuracy: 0.7857
Epoch 5/10
7000/7000 [==============================] - 0s 21us/sample - loss: 0.5131 - accuracy: 0.7937
Epoch 6/10
7000/7000 [==============================] - 0s 20us/sample - loss: 0.5010 - accuracy: 0.7956
Epoch 7/10
7000/7000 [==============================] - 0s 21us/sample - loss: 0.4917 - accuracy: 0.7963
Epoch 8/10
7000/7000 [==============================] - 0s 20us/sample - loss: 0.4844 - accuracy: 0.7960
Epoch 9/10
7000/7000 [==============================] - 0s 21us/sample - loss: 0.4777 - accuracy: 0.7963
Epoch 10/10
7000/7000 [==========

In [21]:
prediction = model.predict(X_test_scale)
out = np.argmax(prediction, 1)

results = confusion_matrix(org_y_test, out)
print("Confusion Matrix:\n", results)

# Accuracy = (TP+TN)/(P+N)
accuracy = (results[0,0] + results[1,1])/(results[0,0] + results[1,0] + results[0,1] + results[1,1])
print("Test accuracy:", accuracy)

Confusion Matrix:
 [[2389    0]
 [ 607    4]]
Test accuracy: 0.7976666666666666


# 7. Optimize the model (Optional)

### a. Find the optimum hyper parameters for  SGD optimizer

In [22]:
def create_model():
    #Initialize Sequential model
    model = tf.keras.models.Sequential()

    # Apply regularization
    model.add(tf.keras.layers.Dropout(0.01))

    #Add Dense Layer to create a hidden layer of 2048 neurons with Relu activation
    model.add(tf.keras.layers.Dense(2048, activation='relu'))

    #Add Dense Layer to create a hidden layer of 128 neurons with tanh activation
    model.add(tf.keras.layers.Dense(128, activation='tanh'))
    
    #Add Dense Layer which provides 2 Outputs after applying softmax
    model.add(tf.keras.layers.Dense(2, activation='softmax'))

    #Create SDG optimizer
    sgd_optimizer = tf.keras.optimizers.SGD(lr=0.01)

    #Comile the model
    model.compile(optimizer=sgd_optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model


model = create_model()
model.fit(X_train_scale, y_train,
          epochs=10,
          batch_size=16)


prediction = model.predict(X_test_scale)
out = np.argmax(prediction, 1)

results = confusion_matrix(org_y_test, out)
print("Confusion Matrix:\n", results)

# Accuracy = (TP+TN)/(P+N)
accuracy = (results[0,0] + results[1,1])/(results[0,0] + results[1,0] + results[0,1] + results[1,1])
print("Test accuracy:", accuracy)

Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 84us/sample - loss: 0.4549 - accuracy: 0.8056
Epoch 2/10
7000/7000 [==============================] - 1s 78us/sample - loss: 0.4149 - accuracy: 0.8283
Epoch 3/10
7000/7000 [==============================] - 1s 74us/sample - loss: 0.3947 - accuracy: 0.8396
Epoch 4/10
7000/7000 [==============================] - 1s 76us/sample - loss: 0.3814 - accuracy: 0.8451
Epoch 5/10
7000/7000 [==============================] - 1s 72us/sample - loss: 0.3705 - accuracy: 0.8516
Epoch 6/10
7000/7000 [==============================] - 1s 77us/sample - loss: 0.3644 - accuracy: 0.8523
Epoch 7/10
7000/7000 [==============================] - 1s 73us/sample - loss: 0.3598 - accuracy: 0.8536
Epoch 8/10
7000/7000 [==============================] - 1s 77us/sample - loss: 0.3592 - accuracy: 0.8526
Epoch 9/10
7000/7000 [==============================] - 1s 73us/sample - loss: 0.3565 - accuracy: 0.8516
Epoch 10/10
7000/7000 [==========

### b. Find the optimum hyper parameters for Adagrad optimizer

In [23]:
def create_model():
    #Initialize Sequential model
    model = tf.keras.models.Sequential()

    # Apply regularization
    model.add(tf.keras.layers.Dropout(0.01))

    #Add Dense Layer to create a hidden layer of 2048 neurons with Relu activation
    model.add(tf.keras.layers.Dense(2048, activation='relu'))

    #Add Dense Layer to create a hidden layer of 128 neurons with tanh activation
    model.add(tf.keras.layers.Dense(128, activation='tanh'))
    
    #Add Dense Layer which provides 10 Outputs after applying softmax
    model.add(tf.keras.layers.Dense(2, activation='softmax'))

    #Create Adam optimizer
    optimizer = tf.keras.optimizers.Adagrad(lr=0.01)

    #Comile the model
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model()
model.fit(X_train_scale, y_train,
          epochs=10,
          batch_size=16)

prediction = model.predict(X_test_scale)
out = np.argmax(prediction, 1)

results = confusion_matrix(org_y_test, out)
print("Confusion Matrix:\n", results)

# Accuracy = (TP+TN)/(P+N)
accuracy = (results[0,0] + results[1,1])/(results[0,0] + results[1,0] + results[0,1] + results[1,1])
print("Test accuracy:", accuracy)

Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 117us/sample - loss: 0.4248 - accuracy: 0.8191
Epoch 2/10
7000/7000 [==============================] - 1s 103us/sample - loss: 0.3814 - accuracy: 0.8446
Epoch 3/10
7000/7000 [==============================] - 1s 97us/sample - loss: 0.3668 - accuracy: 0.8503
Epoch 4/10
7000/7000 [==============================] - 1s 101us/sample - loss: 0.3636 - accuracy: 0.8511
Epoch 5/10
7000/7000 [==============================] - 1s 100us/sample - loss: 0.3584 - accuracy: 0.8501
Epoch 6/10
7000/7000 [==============================] - 1s 101us/sample - loss: 0.3549 - accuracy: 0.8516
Epoch 7/10
7000/7000 [==============================] - 1s 97us/sample - loss: 0.3507 - accuracy: 0.8541
Epoch 8/10
7000/7000 [==============================] - 1s 101us/sample - loss: 0.3486 - accuracy: 0.8569
Epoch 9/10
7000/7000 [==============================] - 1s 99us/sample - loss: 0.3466 - accuracy: 0.8579
Epoch 10/10
7000/7000 [====

### c. Find the optimum hyper parameters for ADAM optimizer

In [24]:
def create_model():
    #Initialize Sequential model
    model = tf.keras.models.Sequential()

    # Apply regularization
    model.add(tf.keras.layers.Dropout(0.01))

    #Add Dense Layer to create a hidden layer of 2048 neurons with Relu activation
    model.add(tf.keras.layers.Dense(2048, activation='relu'))

    #Add Dense Layer to create a hidden layer of 128 neurons with tanh activation
    model.add(tf.keras.layers.Dense(128, activation='tanh'))
    
    #Add Dense Layer which provides 10 Outputs after applying softmax
    model.add(tf.keras.layers.Dense(2, activation='softmax'))

    #Create Adam optimizer
    optimizer = tf.keras.optimizers.Adam(lr=0.001)

    #Comile the model
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model()
model.fit(X_train_scale, y_train,
          epochs=10,
          batch_size=16)


prediction = model.predict(X_test_scale)
out = np.argmax(prediction, 1)

results = confusion_matrix(org_y_test, out)
print("Confusion Matrix:\n", results)

# Accuracy = (TP+TN)/(P+N)
accuracy = (results[0,0] + results[1,1])/(results[0,0] + results[1,0] + results[0,1] + results[1,1])
print("Test accuracy:", accuracy)

Train on 7000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 102us/sample - loss: 0.4122 - accuracy: 0.8293
Epoch 2/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3714 - accuracy: 0.8481
Epoch 3/10
7000/7000 [==============================] - 1s 93us/sample - loss: 0.3590 - accuracy: 0.8513
Epoch 4/10
7000/7000 [==============================] - 1s 89us/sample - loss: 0.3553 - accuracy: 0.8523
Epoch 5/10
7000/7000 [==============================] - 1s 94us/sample - loss: 0.3490 - accuracy: 0.8526
Epoch 6/10
7000/7000 [==============================] - 1s 92us/sample - loss: 0.3483 - accuracy: 0.8584
Epoch 7/10
7000/7000 [==============================] - 1s 90us/sample - loss: 0.3475 - accuracy: 0.8571
Epoch 8/10
7000/7000 [==============================] - 1s 95us/sample - loss: 0.3423 - accuracy: 0.8614
Epoch 9/10
7000/7000 [==============================] - 1s 91us/sample - loss: 0.3412 - accuracy: 0.8596
Epoch 10/10
7000/7000 [=========

In [25]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dropout_2 (Dropout)          multiple                  0         
_________________________________________________________________
dense_8 (Dense)              multiple                  22528     
_________________________________________________________________
dense_9 (Dense)              multiple                  262272    
_________________________________________________________________
dense_10 (Dense)             multiple                  258       
Total params: 285,058
Trainable params: 285,058
Non-trainable params: 0
_________________________________________________________________


### From the optimizations tried above its been conluded that the below hyper parameters giving the max accuracy:
#### Here there is a basic problem of imbalanced classes of data, which gives an impact on accuracy
#### The input layer have 10 neurons
#### 1. Number of Layers - 2 hidden layers with neurons 2048 and 128 with activation layer of relu and tanh
#### 2. Type of optimization used - Adam
#### 3. Learning rate - 0.001
#### 4. Dropout rate - 0.01

# 8. Predict the results using 0.5 as a threshold (Optional)

# 9. Print the Accuracy score and confusion matrix (5 points)

In [26]:
prediction = model.predict(X_test_scale)
out = np.argmax(prediction, 1)

results = confusion_matrix(org_y_test, out)
print("Confusion Matrix:\n", results)

# Accuracy = (TP+TN)/(P+N)
accuracy = (results[0,0] + results[1,1])/(results[0,0] + results[1,0] + results[0,1] + results[1,1])
print("Test accuracy:", accuracy)

Confusion Matrix:
 [[2333   56]
 [ 356  255]]
Test accuracy: 0.8626666666666667
